# 03 - Keyword Evaluation & Discovery Notebook
## Instagram Competition Monitoring & Curation System

Notebook ini dijalankan setiap 4 minggu untuk:
1. Menghitung **Relevance Rate** kumulatif (D8) untuk setiap keyword berdasarkan kurasi admin di Competition Report.
2. Memperbarui metrik dan prioritas pada sheet **Search Keywords** (Total Found, Total Reviewed, Valid Count, Invalid Count, Relevance Rate, Priority, Evaluation Status, Last Evaluated).
3. Menemukan frasa kandidat keyword baru (n-gram 2–4 kata) dari postingan berstatus Valid tanpa AI (D7).
4. Menyimpan kandidat baru sebagai baris baru di sheet **Search Keywords** (Category: `Discovery`, Active Status: `Kandidat`, Priority: `Low`, Frequency: `1`).
5. Menuliskan satu baris ke **Execution Log**.

### 0. Environment Check & Smart Install (wajib dijalankan pertama)
Sel pertama mencetak versi Python dan paket, lalu memeriksa import serta signature API yang dipakai `compmon`.
Sel kedua hanya memasang paket yang **belum ada atau terlalu lama** (tidak pernah `pip install -r`, tidak menurunkan paket bawaan Colab).
Jika ada paket yang dipasang: **Runtime > Restart session**, lalu jalankan ulang notebook dari atas.


In [ ]:
# @title 0a. Environment Check
# Memeriksa Python, versi paket, import, dan signature API yang dipakai compmon.
import sys
from pathlib import Path

_root = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from compmon.env_check import run_environment_check

run_environment_check()


In [ ]:
# @title 0b. Smart Install
# Hanya memasang paket yang belum ada / di bawah versi minimum di requirements.txt.
# Tidak memakai `pip install -r` dan tidak menurunkan paket bawaan Colab.
from compmon.env_check import ensure_environment

ensure_environment()


### 1. Setup Lingkungan & Autentikasi Google Sheets
Memuat kode modul `compmon` (dependensi sudah dicek dan dipasang oleh sel 0) dan melakukan autentikasi via `google.auth.default()` + `gspread.authorize()`.

In [ ]:
# @title 1. Muat Kode
import os, sys
from pathlib import Path

# Pastikan root project dan compmon masuk ke sys.path
root_dir = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from google.colab import auth, userdata
import google.auth
import gspread

auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

SPREADSHEET_ID = userdata.get('SPREADSHEET_ID')

### 2. Jalankan Evaluasi & Discovery Keyword
Menghitung metrik, memperbarui sheet `Search Keywords`, menambahkan baris kandidat baru tanpa mengubah `Active Status` yang ada, dan mencatat ke `Execution Log`.

In [ ]:
from compmon.sheets_repo import SheetsRepository
from compmon.keyword_eval import run_keyword_evaluation

repo = SheetsRepository(gc, SPREADSHEET_ID)

# Seluruh logika evaluasi ada di compmon.keyword_eval (notebook tetap tipis).
summary = run_keyword_evaluation(repo, notebook_name="KEYWORD-EVAL")

print("=== Evaluasi Keyword Selesai ===")
print(f"Keyword dievaluasi : {summary['keywords_written']}")
print(f"Execution ID       : {summary['execution_id']}")
print(f"Kandidat baru      : {len(summary['candidates'])}")
for c in summary["candidates"]:
    print(f"  - {c}")
